# pymefpp_create_associations

This notebook is used to create the associations required to run mefpp


# Testing notebook

It works without any problem or warning

In [1]:
import os
import sys
import glob
import logging

from pathlib import Path

# here we import the local mefpp_run library
sys.path.insert(0, os.getcwd())
import mefpp_associate_reduction as mar

import pycavi_tools
import pycavi_instrument

In [2]:
# Reading configuration file
#
# Asume that all notebooks are in a directory where the data reduction will
# be carried out from. It will contain initially all python libraries and 
# a the config subdirectory.
#
# debug_mode: True
# log_level: info
# observatory: Paranal
# telescope: VST
# instrument: OmegaCam
# file_prefix: OMEGA
# raw_data_dir: /Users/fselman/Pictures/OCAM/CPPup/raw
# project_dir: /Users/fselman/Pictures/OCAM/CPPup2
# reduced_calib_dir: /Users/fselman/Pictures/OCAM/CPPup2/calib/
# reduced_science_dir: /Users/fselman/Pictures/OCAM/CPPup2/science/
# project_logs_dir: /Users/fselman/Pictures/OCAM/CPPup2/logs/
# sex_dir_name: Sextractor_dir
# scamp_dir_name: Scamp_dir
# swarp_dir_name: Swarp_dir
# cal_dir_name: Calibration_dir
#
config_dir = os.getcwd()+'/config/'
config = pycavi_tools.read_pycavi_config(config_dir)
raw_data_dir = config['Directories']['raw_data_dir']
project_dir = config['Directories']['project_dir']
static_cal_dir = config['Directories']['static_calib_dir']
reduced_cal_dir = config['Directories']['reduced_calib_dir']
reduced_science_dir = config['Directories']['project_dir']     # Note that for this notebook we set the reduced science dir to the root project dir.
log_dir = config['Directories']['project_logs_dir']

# Should get these from an instrument configuration file
instrument = pycavi_instrument.Instrument(config_dir, instrument='OCAM')
bpm_prefix = instrument.bpm_prefix
bias_prefix = instrument.bias_prefix
domeff_prefix = instrument.domeff_prefix
skyff_prefix = instrument.skyff_prefix
mff_prefix = instrument.mff_prefix

#create the log_dir outside the science dir.
try:
    os.makedirs(log_dir, exist_ok=True)
except Exception as e:
    print(e)
    pass    


# we prepare the logs
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_dir+"/pymeffpp_associate.log"),
                              logging.StreamHandler()
                    ])

MEF.BIAS.NAME.O: OCAM_MB
MEF.BPM.NAME.O: OCAM_BPM
MEF.DOMEFF.NAME.O: OCAM_DOMEFF
MEF.SKYFF.NAME.O: OCAM_SKYFF
MEF.MFF.NAME.O: OCAM_MFF


In [12]:
def link_bpm(partitions, bpm_partition, instrument, part_type='sci'):
    """
    Creates symbolic links in all partitions in list partitions, to the
    corresponding master BPM file in the static calibration directory.

    Note that the BPM files needs to exists before creating the symbolic links.
    """
    for part in sci_partitions:
        part.bpmPartition = bpm_partition
        #print((part.bpmPartition).base_path)
        #print(instrument.bpm_prefix+'*'+instrument.reduced_file_suffix)
        for i in range(1, instrument.nextensions+1):
            bpm_file = (part.bpmPartition).base_path+'/'+instrument.bpm_prefix+'_%02d.fits'%(i,)
            if not Path(bpm_file).is_file():
                try:
                    os.symlink(bpm_file, part.base_path+'/'+instrument.bpm_prefix+'_%02d.fits'%(i,))
                    print("create symbolic link from : {}".format(part.base_path+'/'+instrument.bpm_prefix+'_%02d.fits'%(i,)))
                    print("                       to : {}".format(bpm_file))
                    Path((part.bpmPartition).base_path+'/toreduce_'+part_type).touch()
                    pass
                except Exception as e:
                    print(e)
                    pass
                
            else:
                print("symbolic link exists from : {}".format(part.base_path+'/'+bpm_filename))
                print("                       to : {}".format(bpm_file))

def select_nearest_calib(partitions, cal_partitions, part_type='bias', ndays_max=10):
    """
    Select the calibration nearest in time. For dome, sky flats, std, and sci we
    match the filter.

    This function does the linking only in memory. The command link needs to be
    invoqued afterward.
    """
    for part in partitions:
        # here we look for the closest in time bias partition
        zdate = part.OBstart

        # we find the bias closest in time
        calBestDeltaT = 3600.*24.*ndays_max
        for cal_part in cal_partitions:
            if 'bias' not in part_type:
                if part.filter != cal_part.filter:
                    continue
            cal_date = cal_part.OBstart
            deltaT = mar.timeDistance(zdate, cal_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < calBestDeltaT:
                calBestDeltaT = deltaT
                if 'bias' in part_type:
                    part.biasPartition = cal_part
                elif 'dome' in part_type:
                    part.domePartition = cal_part
                elif 'sky' in part_type:
                    part.skyPartition = cal_part
                    part.mffPartition = cal_part
                

In [3]:
exposurePartition = mar.PartitionsTree(raw_data_dir, static_cal_dir, reduced_cal_dir, reduced_science_dir, log_dir, instrument)
bpm_partitions = exposurePartition.bpm_partitions
bias_partitions = exposurePartition.bias_partitions
dome_partitions = exposurePartition.dome_partitions
sky_partitions = exposurePartition.sky_partitions
sci_partitions = exposurePartition.sci_partitions

2024-06-28 15:29:14,608:INFO:bias/574184/2017-03-14T09:41:34 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-14T09:50:32.247.fits
2024-06-28 15:29:14,609:INFO:bias/574184/2017-03-14T09:41:34 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-14T09:54:04.247.fits
2024-06-28 15:29:14,610:INFO:bias/574184/2017-03-14T09:41:34 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-14T09:49:07.247.fits
2024-06-28 15:29:14,611:INFO:bias/574184/2017-03-14T09:41:34 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-14T09:53:21.248.fits
2024-06-28 15:29:14,613:INFO:bias/574184/2017-03-14T09:41:34 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-14T09:47:42.877.fits
2024-06-28 15:29:14,614:INFO:bias/574184/2017-03-14T09:41:34 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-14T09:48:25.247.fits
2024-06-28 15:29:14,615:INFO:bias/574184/2017-03-14T09:41:34 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-14T09:52:39.247.fits
2024-06-28 15:29:14,617:INFO:bias/

argument for glob.glob: /Users/fselman/Pictures/OCAM/CPPup/raw//**/OMEGA.*.fits
number of raw files found: 39
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  0 Total     4
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5
BIAS:     2 DOME:     1 SKY:      1 STD:      0 SCIENCE:  1 Total     5


2024-06-28 15:29:14,651:INFO:sky/200253584/r_SDSS/2017-03-13T23:12:13 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-13T23:14:56.159.fits
2024-06-28 15:29:14,653:INFO:sky/200253584/r_SDSS/2017-03-13T23:12:13 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-13T23:16:27.509.fits
2024-06-28 15:29:14,655:INFO:sky/200253584/r_SDSS/2017-03-13T23:12:13 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-13T23:18:00.549.fits
2024-06-28 15:29:14,657:INFO:science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-13T02:40:12.356.fits
2024-06-28 15:29:14,659:INFO:science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-13T02:36:57.286.fits
2024-06-28 15:29:14,661:INFO:science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49 -- /Users/fselman/Pictures/OCAM/CPPup/raw/OMEGA.2017-03-13T02:35:52.825.fits
2024-06-28 15:29:14,663:INFO:science/CP_Pup_081146p0

number of bias partitions:  2
2017-03-13T09:44:19
2017-03-14T09:41:34
number of dome partitions:  1
2017-03-14T09:41:34 r_SDSS
number of sky partitions:  1
2017-03-13T23:12:13 r_SDSS
number of science partitions:  1
2017-03-13T01:57:49 r_SDSS


# We associate a bpm partition to each science frame and create symbolic link in
# corresponding directory
for part in sci_partitions:
    part.bpmPartition = bpm_partitions[0]
    for i in range(1,instrument.next_valid+1):
        bpm_file = (part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((part.bpmPartition).base_path+'/toreduce_sci').touch()
        print(bpm_file, part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))


In [13]:
# We associate a bpm partition to each science frame and create symbolic link in
# corresponding directory
link_bpm(sci_partitions, bpm_partitions[0], instrument, part_type='sci')
link_bpm(dome_partitions, bpm_partitions[0], instrument, part_type='dome')
link_bpm(sky_partitions, bpm_partitions[0], instrument, part_type='sky')
link_bpm(bias_partitions, bpm_partitions[0], instrument, part_type='bias')

create symbolic link from : /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OCAM_BPM_01.fits
                       to : /Users/fselman/Pictures/OCAM/CPPup4//static_calib/bpm/OCAM_BPM_01.fits
create symbolic link from : /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OCAM_BPM_02.fits
                       to : /Users/fselman/Pictures/OCAM/CPPup4//static_calib/bpm/OCAM_BPM_02.fits
create symbolic link from : /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OCAM_BPM_03.fits
                       to : /Users/fselman/Pictures/OCAM/CPPup4//static_calib/bpm/OCAM_BPM_03.fits
create symbolic link from : /Users/fselman/Pictures/OCAM/CPPup4/science/CP_Pup_081146p060_m352105p0/r_SDSS/2017-03-13T01:57:49/OCAM_BPM_04.fits
                       to : /Users/fselman/Pictures/OCAM/CPPup4//static_calib/bpm/OCAM_BPM_04.fits
create symbolic link from : 

In [8]:
# we associate a bias and master flat field partitions to each science partition
def select_nearest_calib(partitions, cal_partitions, part_type='bias', ndays_max=10):
    """
    Select the calibration nearest in time. For dome, sky flats, std, and sci we
    match the filter.

    This function does the linking only in memory. The command link needs to be
    invoqued afterward.
    """
    for part in partitions:
        # here we look for the closest in time bias partition
        zdate = part.OBstart

        # we find the bias closest in time
        calBestDeltaT = 3600.*24.*ndays_max
        for cal_part in cal_partitions:
            if 'bias' not in part_type:
                if part.filter != cal_part.filter:
                    continue
            cal_date = cal_part.OBstart
            deltaT = mar.timeDistance(zdate, cal_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < calBestDeltaT:
                calBestDeltaT = deltaT
                if 'bias' in part_type:
                    part.biasPartition = cal_part
                elif 'dome' in part_type:
                    part.domePartition = cal_part
                elif 'sky' in part_type:
                    part.skyPartition = cal_part
                    part.mffPartition = cal_part
                
for sci_part in sci_partitions:
    # here we look for the closest in time bias partition
    sci_date = sci_part.OBstart

    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(sci_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sci_part.biasPartition = bias_part
    
    # we find the sky closest in time and we link the mff to a file that
    # will be present there once it is created
    #
    skyBestDeltaT = 3600.*24.*30.0
    for sky_part in sky_partitions:
        if sci_part.filter != sky_part.filter:
            continue
        sky_date = sky_part.OBstart
        deltaT = mar.timeDistance(sci_date, sky_date)

        if deltaT < skyBestDeltaT:
            skyBestDeltaT = deltaT
            sci_part.skyPartition = sky_part
            sci_part.mffPartition = sky_part

    
    print('science: %s %s'%(sci_part.filter, sci_part.OBstart,)+\
            '     bias: %s (%3.1f)'%(sci_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' sky: %s %s (%3.1f)'%(sci_part.skyPartition.filter, sci_part.skyPartition.OBstart, skyBestDeltaT/3600.0/24.,))

# at this point we have the biases and mff for each science frame.
# Now we create virtual calibration products as a symbolic link.
#
for sci_part in sci_partitions:
    mbias = (sci_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sci_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((sci_part.biasPartition).base_path+'/toreduce_sci').touch()
        print(mbias)

    mff = (sci_part.mffPartition).base_path+'/'+mff_prefix+'_'+sci_part.filter+'_1x1.fits'
    if not Path(mff).is_file():
        try:
            os.symlink(mff, sci_part.base_path+'/'+mff_prefix+'_'+sci_part.filter+'_1x1.fits')
        except:
            pass
        Path((sci_part.mffPartition).base_path+'/toreduce_mff').touch()
        print(mff)


science: r_SDSS 2017-03-13T01:57:49     bias: 2017-03-13T09:44:19 (0.3) sky: r_SDSS 2017-03-13T23:12:13 (0.9)
/Users/fselman/Pictures/OCAM/CPPup4//calib/bias/574184/2017-03-13T09:44:19/OCAM_MB_1x1.fits
/Users/fselman/Pictures/OCAM/CPPup4//calib/sky/200253584/r_SDSS/2017-03-13T23:12:13/OCAM_MFF_r_SDSS_1x1.fits


#We associate a bpm partition to each sky frame partition
for sky_part in sky_partitions:
    sky_part.bpmPartition = bpm_partitions[0]
    for i in range(1,instrument.next_valid+1):
        bpm_file = (sky_part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sky_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sky_part.bpmPartition).base_path+'/toreduce_sky').touch()
        print(bpm_file, sky_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))

/Users/fselman/Pictures/OCAM/CPPup4//static_calib/bpm
OCAM_BPM*.fits


In [ ]:
# here we associate dome and bias to each sky partition
for sky_part in sky_partitions:
    sky_date = sky_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(sky_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sky_part.biasPartition = bias_part
    
    # we find the dome closest in time and we link the domeff to a file that
    # will be present there once it is created
    #
    domeBestDeltaT = 3600.*24.*30.0
    for dome_part in dome_partitions:
        if sky_part.filter == dome_part.filter:
            dome_date = dome_part.OBstart
            deltaT = mar.timeDistance(sky_date, dome_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < domeBestDeltaT:
                domeBestDeltaT = deltaT
                sky_part.domePartition = dome_part
        else:
            pass

    
    print('sky: %s %s'%(sky_part.filter, sky_part.OBstart,)+\
            '     bias: %s (%4.2f)'%(sky_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' dome: %s %s (%4.2f)'%(sky_part.domePartition.filter, sky_part.domePartition.OBstart, domeBestDeltaT/3600.0/24.,))
    
# at this point we have the biases and domes for each sky ff frame. Now we link
for sky_part in sky_partitions:
    mbias = (sky_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sky_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((sky_part.biasPartition).base_path+'/toreduce_sky').touch()
        
    domeff = (sky_part.domePartition).base_path+'/'+domeff_prefix+'_'+sky_part.filter+'_1x1.fits'
    if not Path(domeff).is_file():
        try:
            os.symlink(domeff, sky_part.base_path+'/'+domeff_prefix+'_'+sky_part.filter+'_1x1.fits')
        except:
            pass
        Path((sky_part.domePartition).base_path+'/toreduce_sky').touch()


In [ ]:
#We associate a bpm partition to each dome ff frame
for dome_part in dome_partitions:
#    dome_part.bpmPartition = bpm_partitions[0]
#    for i in range(1,instrument.next_valid+1):
#       bpm_file = (dome_part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
#        if not Path(bpm_file).is_file():
#            try:
#                os.symlink(bpm_file, dome_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
#                pass
#            except:
#                pass
#            Path((dome_part.bpmPartition).base_path+'/toreduce_dome').touch()

    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(dome_date, bias_date)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part        
    
    mbias = (dome_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()

In [ ]:
# here we associate biases to each dome partition
for dome_part in dome_partitions:
    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(dome_date, bias_date)
        
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part

    
    print('dome: %s %s'%(dome_part.filter, dome_part.OBstart,)+\
            '     bias: %s (%5.3f)'%((dome_part.biasPartition).OBstart, biasBestDeltaT/3600.0/24.,))
    
# at this point we have the biases  for each dome ff frame. Now we link
for dome_part in dome_partitions:
    mbias = (dome_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()
        

In [ ]:
# Finally associate a bpm partition to each bias frame
for bias_part in bias_partitions:
    bias_part.bpmPartition = bpm_partitions[0]
    for i in range(1,instrument.next_valid+1):
        bpm_file = (bias_part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, bias_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((bias_part.bpmPartition).base_path+'/toreduce_bias').touch()
